# 17 — E0 Pilot v2: Continuous Expression Metric, k ∈ {3, 4} (Step 9 re-run)

Re-run of the E0 pilot with the lessons of notebook 16:

- **Metric v2** — continuous concept-expression: post-hoc projection of each generated
  continuation onto each concept (`fur.project`), reported as the **delta** vs the
  unsteered baseline continuation of the same prompt. Presence-success floored at 0 in
  v1 while steering visibly worked; projection sees topic alignment without requiring
  exact member lemmas, and is method-agnostic.
- **k ∈ {3, 4}** as an explicit factor (fluency cost roughly doubled from k=3 to k=4
  in earlier runs; success may trade off the same way).
- steps=24 (gentler than v1's 32), single batch order (v1: order delta = 0.000).

Matrix: 3 ρ strata × {F1.a, F2.a} × k ∈ {3,4} × 20 prompts = 240 records.

Joint measure: per-concept deltas are z-scored across all records of that concept
(projection scales are not comparable across concepts), then joint = min(z_a, z_b) —
AND semantics on the standardized scale.

Outputs: `resources/17_e0_pilot_v2.jsonl`, `resources/17_e0_summary_v2.csv`,
`resources/17_e0_gap_vs_rho_v2.png`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.representations import FrameUnembeddingRepresentation
from frames.representations.concept import Concept

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
    device_map="cuda:0",
    torch_dtype=torch.float16,
)

MIN_LEMMAS = 11
MAX_TOKENS = 3
KS = (3, 4)
STEPS = 24

SELECTION = json.loads(Path("resources/15_e0_selected_pairs.json").read_text())["selected"]
for stratum, info in SELECTION.items():
    print(f"{stratum:>6}: {info['a']} / {info['b']}  rho={info['rho']:.3f}")

In [ ]:
def chat(user: str, assistant: str = "") -> str:
    return (
        f"<|start_header_id|>user<|end_header_id|>{user}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{assistant}"
    )


PROMPTS = [
    chat("What men can be?", "1."),
    chat("What women can be?", "1."),
    chat("Tell me a short story.", "Once"),
    chat("Describe your favorite place.", "My favorite place is"),
    chat("What do you like to do on weekends?", "I like"),
    chat("Give me some advice for a good life.", "1."),
    chat("What is the most important thing in the world?", "The most important thing is"),
    chat("Describe the weather today.", "Today the weather is"),
    chat("What should I cook for dinner?", "You should"),
    chat("Name things that make people happy.", "1."),
    chat("What did you do today?", "Today I"),
    chat("Describe a memorable meal.", "The most memorable meal"),
    chat("What is a good gift idea?", "A good gift"),
    chat("Describe your hometown.", "My hometown is"),
    chat("Tell me about your favorite season.", "My favorite season is"),
    chat("What makes a good friend?", "A good friend"),
    chat("Describe a place you would like to visit.", "I would like to visit"),
    chat("What do you do to relax?", "To relax, I"),
    chat("Tell me something interesting.", "Here is something interesting:"),
    chat("Describe your morning routine.", "My morning starts"),
]

LOG_PATH = Path("resources/17_e0_pilot_v2.jsonl")
LOG_PATH.unlink(missing_ok=True)

harness = EvaluationHarness(
    fur, LOG_PATH, min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS
)
baseline_texts = harness.generate_baseline(PROMPTS, max_new_tokens=STEPS + 2)
print("baseline ready")

## The matrix

In [ ]:
for stratum, info in SELECTION.items():
    concept_a = fur.get_concept_cached(info["a"], MIN_LEMMAS, MAX_TOKENS)
    concept_b = fur.get_concept_cached(info["b"], MIN_LEMMAS, MAX_TOKENS)
    mean_frame = Concept.average([concept_a, concept_b])
    synsets = [info["a"], info["b"]]
    concepts = {info["a"]: concept_a, info["b"]: concept_b}

    for k in KS:
        texts_f1, probe_f1 = fur.generate_with_topk_guide(
            PROMPTS, guide=mean_frame, k=k, steps=STEPS
        )
        harness.evaluate(
            PROMPTS, texts_f1, synsets,
            config={"stratum": stratum, "rho": info["rho"], "method": "F1.a", "k": k, "steps": STEPS},
            probe=probe_f1, baseline_texts=baseline_texts, concepts=concepts,
        )

        texts_f2, probe_f2 = fur.generate_with_topk_multi_guide(
            PROMPTS, guides=[concept_a, concept_b], k=k, steps=STEPS, normalize="zscore"
        )
        harness.evaluate(
            PROMPTS, texts_f2, synsets,
            config={"stratum": stratum, "rho": info["rho"], "method": "F2.a", "k": k, "steps": STEPS},
            probe=probe_f2, baseline_texts=baseline_texts, concepts=concepts,
        )
        print(f"{stratum} / k={k}: done")

In [ ]:
parsed = [json.loads(line) for line in LOG_PATH.read_text().strip().split("\n")]
assert len(parsed) == 3 * 2 * len(KS) * len(PROMPTS), "one record per cell per prompt"
assert all(r["expression"] is not None for r in parsed), "metric v2 present everywhere"
print(f"gate: {len(parsed)} records parse, expression logged")

## Metric v2 sanity: does steering raise expression at all?

Mean steered vs baseline expression, pooled over both concepts of each pair.

In [ ]:
rows = []
for record in parsed:
    config = record["config"]
    info = SELECTION[config["stratum"]]
    expr = record["expression"]
    rows.append(
        {
            "stratum": config["stratum"],
            "rho": config["rho"],
            "method": config["method"],
            "k": config["k"],
            "concept_a": info["a"],
            "concept_b": info["b"],
            "delta_a": expr[info["a"]]["delta"],
            "delta_b": expr[info["b"]]["delta"],
            "steered_a": expr[info["a"]]["steered"],
            "baseline_a": expr[info["a"]]["baseline"],
            "steered_b": expr[info["b"]]["steered"],
            "baseline_b": expr[info["b"]]["baseline"],
            "ppl_ratio": record["ppl_ratio"],
            "fluency_flag": record["fluency_flag"],
            "success_both": all(record["success"].values()),
        }
    )
results = pd.DataFrame(rows)

sanity = results[["delta_a", "delta_b"]].mean()
print("mean expression delta (steered - baseline):")
print(sanity.round(4))
print("\nshare of records with positive delta:")
print(((results[["delta_a", "delta_b"]] > 0).mean()).round(3))

## Joint expression gain (standardized min) and summary

In [ ]:
# z-score each concept's delta across all its records (per-concept scales differ),
# then joint = min of the two z-scores (AND semantics)
for side in ("a", "b"):
    zs = []
    for concept, group in results.groupby(f"concept_{side}")[f"delta_{side}"]:
        z = (group - group.mean()) / (group.std() + 1e-8)
        zs.append(z)
    results[f"z_{side}"] = pd.concat(zs).sort_index()
results["joint_z"] = results[["z_a", "z_b"]].min(axis=1)

summary = (
    results.groupby(["stratum", "rho", "method", "k"])
    .agg(
        delta_a=("delta_a", "mean"),
        delta_b=("delta_b", "mean"),
        joint_z=("joint_z", "mean"),
        ppl_ratio=("ppl_ratio", "mean"),
        flag_share=("fluency_flag", "mean"),
        presence_both=("success_both", "mean"),
    )
    .reset_index()
    .sort_values(["k", "rho", "method"])
)
summary.to_csv("resources/17_e0_summary_v2.csv", index=False)
summary.round(4)

## THE plot v2 — gap vs ρ, per k

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
styles = {3: "-", 4: "--"}

for k in KS:
    sub = summary[summary["k"] == k]
    f1 = sub[sub["method"] == "F1.a"].set_index("rho").sort_index()
    f2 = sub[sub["method"] == "F2.a"].set_index("rho").sort_index()
    rhos = list(f1.index)

    axes[0].plot(rhos, f1["joint_z"], "o" + styles[k], color="tab:blue", label=f"F1.a k={k}")
    axes[0].plot(rhos, f2["joint_z"], "s" + styles[k], color="tab:orange", label=f"F2.a k={k}")

    axes[1].plot(rhos, (f2["joint_z"] - f1["joint_z"]), "d" + styles[k], color="tab:red", label=f"k={k}")

    axes[2].plot(rhos, f1["ppl_ratio"], "o" + styles[k], color="tab:blue", label=f"F1.a k={k}")
    axes[2].plot(rhos, f2["ppl_ratio"], "s" + styles[k], color="tab:orange", label=f"F2.a k={k}")

axes[0].set_title("joint expression gain (standardized min)")
axes[0].set_ylabel("mean joint z")
axes[1].set_title("THE gap: F2.a - F1.a joint gain vs rho")
axes[1].axhline(0, color="gray", lw=0.5)
axes[2].set_title("fluency cost")
axes[2].set_ylabel("mean PPL ratio")
axes[2].axhline(2.5, color="gray", ls="--", lw=0.8)
for ax in axes:
    ax.set_xlabel("rho")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("resources/17_e0_gap_vs_rho_v2.png", dpi=150)
plt.show()

## Examples

In [ ]:
def answer(text: str) -> str:
    return text.split("assistant<|end_header_id|>")[-1]


for stratum in SELECTION:
    for method in ("F1.a", "F2.a"):
        example = next(
            r for r in parsed
            if r["config"]["stratum"] == stratum
            and r["config"]["method"] == method
            and r["config"]["k"] == 3
            and "short story" in r["prompt"]
        )
        deltas = {
            name.split(".")[0]: round(v["delta"], 4)
            for name, v in example["expression"].items()
        }
        print(f"[{stratum} / {method} / k=3] deltas={deltas}")
        print("   ", answer(example["text"])[:130])
    print()